# 🎓 Szkolny asystent AI: nauczmy model Twoich notatek i regulaminu (RAG)

**Warsztaty dla uczniów szkół średnich** · Google Colab (darmowa maszyna **T4 GPU**)

Zbudujemy bota, który odpowiada na pytania o **regulamin szkoły i notatki z lekcji** — i potrafi pokazać, skąd wziął odpowiedź. Tę samą technikę możesz potem zastosować do własnych notatek na sprawdzian.

---

### Dlaczego w ogóle musimy „doszkalać" modele?

Duży model językowy (LLM, np. ChatGPT, Gemma, Llama) uczył się na ogromnej ilości tekstów z internetu — ale tylko do pewnego dnia.
To oznacza, że model:

* ❌ **nie zna wydarzeń z ostatnich miesięcy**,
* ❌ **nie zna Twoich prywatnych danych** (notatek, regulaminu Twojej szkoły, dokumentów firmy),
* ❌ **potrafi zmyślać** (to się nazywa *halucynacje*) — i robi to bardzo pewnym siebie tonem!

### Dwa sposoby, żeby to naprawić

| | **Fine-tuning** (douczanie wag) | **RAG** (Retrieval-Augmented Generation) |
|---|---|---|
| Analogia | uczysz się do egzaminu **na pamięć** | piszesz egzamin z **otwartymi notatkami** |
| Co się zmienia? | model sam się zmienia (liczby w jego „mózgu") | model zostaje taki sam, dostaje „ściągę" w pytaniu |
| Koszt | duży (GPU, czas, dane) | mały (wystarczy laptop) |
| Aktualizacja wiedzy | trzeba trenować od nowa | wystarczy dopisać dokument |
| Źródło odpowiedzi | trudno sprawdzić | **możemy pokazać, skąd model to wziął** |

> **RAG = Retrieval (wyszukaj) + Augmented (wzbogać) + Generation (wygeneruj)**

### Plan warsztatu

```
 Pytanie ──► [1. WYSZUKAJ] ──► fragment wiedzy ──► [2. WKLEJ DO PROMPTU] ──► [3. LLM ODPOWIADA]
              ▲  (embeddingi)                                                      │
              │                                                                    ▼
        Baza wiedzy                                                       Odpowiedź ze źródłem
```

* **Część 1 – Prosty RAG od zera** (bez baz danych, kilkanaście linijek kodu) — zobaczysz, że działa.
* **Część 2 – Embeddingi: serce wyszukiwarki** — zajrzymy do środka i sprawdzimy, jak komputer „rozumie" znaczenie zdań.
* **Część 3 – Podsumowanie i zadania na własną rękę.**

> 💡 W notatniku znajdziesz komórki oznaczone **🧪 Twoja kolej** — to zadania do samodzielnego wykonania.

## Krok 0 – Przygotowanie środowiska

1. Włącz GPU: menu **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU**.
2. Uruchamiaj komórki po kolei (**Shift + Enter**).

**Model Gemma wymaga zgody na licencję.** Prowadzący przygotował token Hugging Face — jeśli pracujesz na własnym koncie:
załóż konto na [huggingface.co](https://huggingface.co), zaakceptuj licencję na stronie modelu
[google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it), wygeneruj token (*Settings → Access Tokens*)
i dodaj go w Colabie w zakładce 🔑 **Secrets** pod nazwą `HF_TOKEN` (z włączonym dostępem dla notatnika).

In [ ]:
!pip install -q transformers accelerate sentence-transformers

import os
import torch
from google.colab import userdata

# Token Hugging Face z bezpiecznych zmiennych Colaba (zakładka 🔑 Secrets)
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ Token HF załadowany pomyślnie!")
except Exception:
    print("⚠️ Nie znaleziono tokenu 'HF_TOKEN' w Secrets. Jeśli model zgłosi błąd, dodaj go.")

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ Brak GPU! Wybierz: Środowisko wykonawcze → Zmień typ środowiska → T4 GPU")

# CZĘŚĆ 1 – Prosty RAG od zera

## Krok 1 – Ładujemy model językowy (LLM)

Używamy **Gemma 4 E2B** od Google — to „mały" model (ok. 2 mld efektywnych parametrów), który mieści się w pamięci darmowej karty T4.
Ładowanie trwa kilka minut — to dobry moment na kawę ☕ (lub aeropress, patrz niżej 😉).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E2B-it"
# Gdyby Gemma sprawiała kłopoty z dostępem, można użyć modelu bez bramki licencyjnej:
# model_id = "Qwen/Qwen2.5-3B-Instruct"

print("Ładowanie tokenizera...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Ładowanie modelu (to może chwilę potrwać)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Model załadowany na GPU!")

Zdefiniujmy funkcję, która wysyła pytanie do modelu i zwraca jego odpowiedź.
Dzięki temu później nie będziemy powtarzać tego samego kodu.

In [ ]:
def ask_llm(prompt, max_new_tokens=150):
    """Wysyła prompt do modelu i zwraca jego odpowiedź jako tekst."""
    messages = [{"role": "user", "content": prompt}]

    # Zamiana rozmowy na format, który model rozumie (znaczniki rozmowy)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # bez losowości: te same pytanie = ta sama odpowiedź
        )

    # Odcinamy z wyniku nasze pytanie, zostawiamy tylko odpowiedź modelu
    input_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

print(ask_llm("Podaj stolicę Polski jednym zdaniem."))

## Krok 2 – Nasza baza wiedzy (której model NIE zna)

Mamy **regulamin wymyślonego liceum w Bajtowie** (zasady spóźnień, sprawdzianów, telefonów…), informacje o szkole oraz dwie notatki z lekcji.
Regulaminu tej szkoły żaden model nie mógł zobaczyć w internecie, bo… ona nie istnieje. Dzięki temu wiemy na pewno: jeśli model odpowie poprawnie, to dlatego, że dostał od nas wiedzę, a nie dlatego, że ją „pamiętał".

> W zadaniach dodasz swoje własne dokumenty — np. prawdziwy regulamin Twojej szkoły albo notatki z lekcji.

In [ ]:
# Nasza mini-baza wiedzy: każdy element listy to jeden dokument (tu: jedno lub dwa zdania)
documents = [
    "Regulamin: uczeń, który spóźni się na lekcję więcej niż 15 minut, otrzymuje w dzienniku nieobecność. Spóźnienia do 15 minut są odnotowywane osobno.",
    "Regulamin: nieobecność trzeba usprawiedliwić w ciągu 7 dni od powrotu do szkoły – przez dziennik elektroniczny lub pisemnie (kartka od rodzica).",
    "Regulamin: w czasie lekcji telefon musi być wyłączony i schowany w plecaku. Nauczyciel może pozwolić na jego użycie do konkretnego zadania.",
    "Regulamin: sprawdzian musi być zapowiedziany z tygodniowym wyprzedzeniem. W jednym dniu mogą odbyć się najwyżej dwa sprawdziany.",
    "Regulamin: ocenę ze sprawdzianu można poprawić tylko jeden raz, w ciągu dwóch tygodni od jej otrzymania.",
    "Biblioteka szkolna jest czynna od 7:30 do 15:30. Można wypożyczyć 4 książki na 21 dni i przedłużyć termin jeden raz.",
    "Obiady w stołówce są wydawane od 11:30 do 14:00. Obiad można odwołać najpóźniej do godziny 8:00 w tym samym dniu.",
    "Koło informatyczne spotyka się we wtorki o 15:00 w sali 12. Prowadzi je mgr Anna Bajtowska, a udział jest dobrowolny.",
    "Próbna matura z matematyki odbędzie się w pierwszym tygodniu grudnia i potrwa 180 minut. Wolno używać kalkulatora prostego i tablic matematycznych.",
    "Notatka z biologii: mitochondrium to organellum komórkowe, w którym powstaje ATP w procesie oddychania komórkowego. Nazywa się je „elektrownią komórki”.",
    "Notatka z historii: Mieszko I przyjął chrzest w 966 roku w obrządku łacińskim, za pośrednictwem Czech. Wydarzenie to wprowadziło Polskę do kręgu zachodniego chrześcijaństwa.",
]

print(f"Załadowano {len(documents)} dokumentów do bazy wiedzy.")

## Krok 3 – Eksperyment kontrolny: model **bez** RAG

Zapytajmy model o coś, czego nie może wiedzieć. Zobacz, co zrobi.

In [ ]:
pytanie = "Ile mam dni na usprawiedliwienie nieobecności w naszej szkole?"

print("Pytanie:", pytanie)
print("Odpowiedź BEZ RAG:", ask_llm(pytanie))

**Co zaobserwowałeś/aś?** Czy model przyznał, że nie wie, czy raczej coś zmyślił? Zapisz swoje obserwacje albo porównaj z koleżanką/kolegą — możecie dostać różne wyniki, jeśli zadacie pytanie inaczej.

> 💡 To jest właśnie problem, który rozwiązuje RAG.

## Krok 4 – Retriever: wyszukiwarka znaczeniowa

Jak model ma znaleźć właściwy dokument? Nie możemy po prostu szukać identycznych słów — pytanie i odpowiedź często używają **różnych słów** na to samo.

Rozwiązanie: **embeddingi**. Specjalny, mały model zamienia każdy tekst na **listę liczb** (wektor), tak że teksty o **podobnym znaczeniu** dostają **podobne liczby**. Wtedy wystarczy porównać liczby.

Na razie traktujemy model embeddingów jak czarną skrzynkę — w Części 2 zajrzymy do środka.

Używamy modelu **wielojęzycznego** (rozumie też polski): `paraphrase-multilingual-MiniLM-L12-v2`.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Mały model zamieniający tekst na wektory liczb (działa też po polsku)
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Zamieniamy wszystkie dokumenty na wektory. normalize_embeddings=True skraca każdy wektor do długości 1,
# dzięki czemu podobieństwo policzymy zwykłym iloczynem skalarnym.
doc_embeddings = embed_model.encode(documents, normalize_embeddings=True)

print("Kształt macierzy embeddingów:", doc_embeddings.shape, "→ (liczba dokumentów, liczba liczb w wektorze)")

In [ ]:
def search(query, docs, doc_embs, top_k=1):
    """Zwraca listę (podobieństwo, dokument) dla top_k najbardziej pasujących dokumentów."""
    query_emb = embed_model.encode([query], normalize_embeddings=True)

    # Podobieństwo pytania do każdego dokumentu (im bliżej 1, tym bardziej podobne znaczeniowo)
    scores = np.dot(doc_embs, query_emb.T).flatten()

    # Indeksy dokumentów posortowane od najlepszego do najgorszego
    best = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), docs[i]) for i in best]

# Szybki test wyszukiwarki
for score, doc in search("Co grozi za spóźnienie na lekcję?", documents, doc_embeddings, top_k=3):
    print(f"{score:.2f} | {doc[:90]}...")

Widzisz? Wyszukiwarka sama ustawiła dokumenty od najbardziej do najmniej pasującego — mimo że pytanie i regulamin używają trochę innych słów.

## Krok 5 – Składamy RAG w całość

Teraz łączymy: **wyszukaj → wklej do promptu → wygeneruj odpowiedź**.

Najważniejszy jest **prompt** — instrukcja, w której mówimy modelowi: *„odpowiadaj TYLKO na podstawie tego kontekstu"*.

In [ ]:
def ask_rag(query, docs, doc_embs, top_k=1, min_score=None, verbose=True):
    """Pełny RAG: wyszukaj kontekst → zbuduj prompt → zapytaj model.

    min_score – jeśli najlepszy dokument ma podobieństwo niższe niż ta wartość,
                uznajemy, że w bazie nie ma odpowiedzi (nie pytamy nawet modelu).
    """
    # 1. WYSZUKAJ
    found = search(query, docs, doc_embs, top_k=top_k)
    if verbose:
        for score, doc in found:
            print(f"   🔎 podobieństwo {score:.2f}: {doc[:80]}...")

    if min_score is not None and found[0][0] < min_score:
        return "Nie wiem (nie znalazłem nic podobnego w bazie wiedzy)."

    context = "\n".join(doc for _, doc in found)

    # 2. WKLEJ DO PROMPTU
    prompt = (
        "Odpowiedz na pytanie użytkownika, opierając się TYLKO na podanym kontekście. "
        "Jeśli w kontekście nie ma odpowiedzi, napisz 'Nie wiem'.\n\n"
        f"Kontekst: {context}\n\n"
        f"Pytanie: {query}"
    )

    # 3. WYGENERUJ ODPOWIEDŹ
    return ask_llm(prompt)

pytanie = "Ile mam dni na usprawiedliwienie nieobecności w naszej szkole?"
print("Pytanie:", pytanie)
print("Odpowiedź Z RAG:", ask_rag(pytanie, documents, doc_embeddings))

Porównaj z odpowiedzią z Kroku 3. Ta sama sieć neuronowa, to samo pytanie — a odpowiedź zupełnie inna, bo dostała „ściągę".

### Testujemy nasz RAG na różnych pytaniach

In [ ]:
pytania = [
    "Co się stanie, jeśli spóźnię się na lekcję o pół godziny?",
    "Ile razy mogę poprawiać sprawdzian?",
    "Czy mogę używać telefonu na lekcji?",
    "Do której mogę odwołać obiad?",
    "Ile trwa próbna matura z matematyki?",
    "Jaka będzie jutro pogoda w Bajtowie?",   # <- tego nie ma w bazie!
    "Ile kosztuje obiad w stołówce?",         # <- temat jest w bazie, ale ceny nie ma!
]

for p in pytania:
    print("❓", p)
    print("💬", ask_rag(p, documents, doc_embeddings))
    print("-" * 60)

Zwróć uwagę na dwa ostatnie pytania. **O pogodzie w bazie nie ma nic**, a **o cenie obiadu** — choć obiady w bazie są — nikt nie podał kwoty. Dobry RAG powinien w obu przypadkach przyznać się („Nie wiem"), a nie zmyślać.
Dlatego w prompcie napisaliśmy: *„Jeśli w kontekście nie ma odpowiedzi, napisz 'Nie wiem'"*.

### 🧪 Twoja kolej (zadania do Części 1)

1. **Dopisz własny dokument** do listy `documents` (np. zasady z Twojej prawdziwej szkoły albo notatka z lekcji), zrób `doc_embeddings = embed_model.encode(...)` jeszcze raz i zadaj pytanie o ten fakt.
   *To jest cała „aktualizacja wiedzy" w RAG — bez trenowania modelu!*
2. **Zmień instrukcję w prompcie** — np. usuń zdanie „opierając się TYLKO na kontekście". Czy model zaczyna zmyślać?
3. **Zmień `top_k`** na 3. Czy odpowiedzi są lepsze, czy gorsze? (Więcej kontekstu = więcej szansy na właściwą informację, ale też więcej „szumu").
4. **Próg pewności:** wywołaj `ask_rag(pytanie, documents, doc_embeddings, min_score=0.4)`. Funkcja odrzuca pytania, dla których najlepszy dokument ma podobieństwo niższe niż `min_score`.
   Porównaj podobieństwa (`🔎`) dla pytań **z bazy**, pytania o pogodę i pytania o cenę obiadu. Który z „trudnych" przypadków próg wyłapie, a którego nie? *(Podpowiedź: przy cenie obiadu dokument jest tematycznie bardzo bliski, więc próg nie pomoże — tu ratuje nas tylko instrukcja w prompcie „odpowiadaj tylko z kontekstu".)*

In [ ]:
# Miejsce na Twoje eksperymenty ✍️

# CZĘŚĆ 2 – Embeddingi: jak komputer „rozumie" znaczenie?

W Części 1 wyszukiwarka zadziałała „magicznie". Czas zajrzeć pod maskę.

> **Embedding** = tekst zamieniony na punkt w wielowymiarowej przestrzeni. Teksty o podobnym znaczeniu leżą **blisko siebie**, a o różnym — **daleko**.

Analogia: wyobraź sobie mapę, na której każde zdanie ma swoje współrzędne GPS. Zdania o kotach mieszkają w jednej dzielnicy, o pizzy — w innej. Tylko że ta mapa nie ma 2 wymiarów, a kilkaset.

## 2.1 Jak wygląda embedding?

In [ ]:
wektor = embed_model.encode("Kot śpi na kanapie.")

print("Długość wektora:", len(wektor))
print("Pierwsze 10 liczb:", np.round(wektor[:10], 3))

Zdanie zostało zamienione na **listę 384 liczb**. Pojedyncza liczba nic nam nie mówi (to nie jest „liczba kotów" ani „poziom senności") — dopiero **porównanie wektorów** ma sens.

## 2.2 Podobieństwo kosinusowe

Do porównywania wektorów używamy **podobieństwa kosinusowego** — mierzy ono, jak bardzo dwa wektory „patrzą w tę samą stronę":

* **1.0** – identyczny kierunek (to samo znaczenie),
* **0.0** – brak związku,
* **ujemne** – przeciwstawne.

Ponieważ nasze wektory są znormalizowane (długość 1), wystarczy iloczyn skalarny.

In [ ]:
zdania = [
    "Kot śpi na kanapie.",
    "Mruczący kociak drzemie na sofie.",      # to samo znaczenie, inne słowa!
    "Pies goni piłkę w parku.",
    "Kwantowy procesor działa w temperaturze pokojowej.",
    "Samochód elektryczny ładuje się na stacji.",
]

emb = embed_model.encode(zdania, normalize_embeddings=True)

print("Podobieństwo zdania 0 do pozostałych:")
for i in range(1, len(zdania)):
    sim = float(np.dot(emb[0], emb[i]))
    print(f"  {sim:5.2f}  ← {zdania[i]}")

Zdanie 1 nie ma **żadnego wspólnego słowa** z zdaniem 0 (kot/kociak, śpi/drzemie, kanapie/sofie), a mimo to model uznał je za najbardziej podobne. To jest siła embeddingów.

Zobaczmy **całą macierz podobieństw** jako mapę ciepła:

In [ ]:
import matplotlib.pyplot as plt

sim_matrix = np.dot(emb, emb.T)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(sim_matrix, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(zdania)))
ax.set_yticks(range(len(zdania)))
ax.set_xticklabels([f"zd. {i}" for i in range(len(zdania))])
ax.set_yticklabels([z[:30] + "…" for z in zdania])
for i in range(len(zdania)):
    for j in range(len(zdania)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", color="white", fontsize=9)
plt.colorbar(im, label="podobieństwo kosinusowe")
ax.set_title("Które zdania są do siebie podobne?")
plt.tight_layout()
plt.show()

### 🧪 Twoja kolej
Wpisz 3 własne pary zdań:
* parę o **tym samym znaczeniu**, ale innych słowach,
* parę o **podobnych słowach**, ale **przeciwnym** znaczeniu (np. „Lubię matematykę" / „Nie lubię matematyki"),
* parę **całkiem niezwiązaną**.

Który wynik Cię zaskoczył?

In [ ]:
para_a = "Lubię matematykę."
para_b = "Nie lubię matematyki."

a, b = embed_model.encode([para_a, para_b], normalize_embeddings=True)
print(f"Podobieństwo: {float(np.dot(a, b)):.2f}")

## 2.3 Mapa znaczeń – rysujemy embeddingi

Wektor ma 384 liczby, a ekran tylko 2 wymiary. Użyjemy metody **PCA**, która „spłaszcza" dane do 2D, starając się zachować jak najwięcej informacji o odległościach.

Weźmiemy zdania z 4 przedmiotów szkolnych i zobaczymy, czy model sam je pogrupuje (nie mówiliśmy mu, jakie są przedmioty!).

In [ ]:
from sklearn.decomposition import PCA

kategorie = {
    "matematyka": [
        "Pole kwadratu to bok razy bok.", "Rozwiąż równanie kwadratowe za pomocą delty.",
        "Suma kątów w trójkącie wynosi 180 stopni.", "Funkcja liniowa ma wykres w kształcie prostej.",
        "Oblicz pochodną tej funkcji.",
    ],
    "biologia": [
        "Chloroplasty prowadzą fotosyntezę.", "DNA zawiera informację genetyczną.",
        "Serce pompuje krew do całego ciała.", "Mitochondria produkują energię dla komórki.",
        "Żaby są płazami.",
    ],
    "historia": [
        "Bitwa pod Grunwaldem odbyła się w 1410 roku.", "Konstytucja 3 maja została uchwalona w 1791 roku.",
        "Napoleon przegrał pod Waterloo.", "Mieszko I przyjął chrzest w 966 roku.",
        "II wojna światowa zaczęła się w 1939 roku.",
    ],
    "informatyka": [
        "Pętla for powtarza instrukcje wiele razy.", "Zmienna przechowuje wartość w pamięci.",
        "Algorytm sortowania porządkuje listę liczb.", "Błąd w programie nazywamy bugiem.",
        "Sieć neuronowa uczy się na danych.",
    ],
}

wszystkie_zdania = [z for lista in kategorie.values() for z in lista]
etykiety = [k for k, lista in kategorie.items() for _ in lista]

emb_kat = embed_model.encode(wszystkie_zdania, normalize_embeddings=True)
punkty = PCA(n_components=2, random_state=0).fit_transform(emb_kat)

kolory = {"matematyka": "tab:blue", "biologia": "tab:green", "historia": "tab:orange", "informatyka": "tab:red"}
fig, ax = plt.subplots(figsize=(9, 6.5))
for kat, kolor in kolory.items():
    idx = [i for i, e in enumerate(etykiety) if e == kat]
    ax.scatter(punkty[idx, 0], punkty[idx, 1], c=kolor, s=90, label=kat, alpha=0.8)
    for i in idx:
        ax.annotate(wszystkie_zdania[i].split()[0], punkty[i], textcoords="offset points",
                    xytext=(6, 4), fontsize=8)
ax.set_title("Mapa znaczeń zdań (embeddingi spłaszczone do 2D)")
ax.set_xlabel("wymiar 1"); ax.set_ylabel("wymiar 2")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Zdania o tym samym przedmiocie zazwyczaj tworzą **skupiska** — choć model nigdy nie dostał etykiet. Nauczył się tego, czytając ogromne ilości tekstu.

Uwaga: to tylko **cień** prawdziwej, 384-wymiarowej przestrzeni — czasem coś się nakłada na rysunku, choć w pełnej przestrzeni jest daleko.

### 🧪 Twoja kolej
Dodaj do jednej z kategorii nowe zdanie albo wymyśl piąty przedmiot (np. „geografia", „chemia"). Czy nowe punkty trafiają w sensowne miejsce? Spróbuj też dodać zdanie „na granicy" dwóch tematów (np. „Oblicz statystycznie, ile mitochondriów ma komórka"). Gdzie wyląduje?

## 2.4 Wyszukiwanie po znaczeniu a wyszukiwanie po słowach

Porównajmy naszą wyszukiwarkę z klasycznym wyszukiwaniem po słowach kluczowych (takim, jak `Ctrl+F`).

In [ ]:
import re

def slowa(tekst):
    return set(re.findall(r"\w+", tekst.lower()))

def search_keywords(query, docs):
    """Wyszukiwanie 'Ctrl+F': wygrywa dokument z największą liczbą wspólnych słów."""
    q = slowa(query)
    wspolne = [len(q & slowa(d)) for d in docs]
    best = int(np.argmax(wspolne))
    return (docs[best], wspolne[best])

pytania_test = [
    "Do kiedy rodzic ma wytłumaczyć moją absencję?",  # w regulaminie: 'usprawiedliwić nieobecność'
    "Do której mogę zrezygnować z obiadu?",    # w regulaminie: 'odwołać', 'obiad'
    "Skąd komórka bierze energię?",             # w notatce: 'mitochondrium', 'ATP'
]

for p in pytania_test:
    print("❓", p)
    d_kw, n = search_keywords(p, documents)
    print(f"   📝 słowa kluczowe ({n} wspólnych słów): {d_kw[:70]}...")
    score, d_sem = search(p, documents, doc_embeddings, top_k=1)[0]
    print(f"   🧠 znaczeniowo ({score:.2f}):            {d_sem[:70]}...")
    print()

Wyszukiwanie po słowach jest „kruche": polski ma odmianę (*komórka / komórkowe / komórki*), mamy synonimy (*telefon / smartfon / komórka*), parafrazy… Embeddingi radzą sobie z tym znacznie lepiej, bo porównują **znaczenie**, nie litery.

## 2.5 Nie każdy model embeddingów jest równie dobry

Różne modele embeddingów uczono na różnych danych. Model uczony głównie na **angielskim** może słabo radzić sobie z **polskim**. Sprawdźmy to eksperymentalnie!

Zrobimy mały **test jakości**: zestaw pytań, dla których wiemy, który dokument jest właściwy. Sprawdzimy, jak często model wskaże go jako najlepszy.

In [ ]:
# Pytania testowe i numer właściwego dokumentu na liście `documents` (liczymy od 0)
testy = [
    ("Co się stanie, jeśli przyjdę na lekcję pół godziny po dzwonku?", 0),
    ("Do kiedy rodzic ma wytłumaczyć moją absencję?", 1),
    ("Czy mogę mieć smartfon na ławce podczas lekcji?", 2),
    ("Ile klasówek może być jednego dnia?", 3),
    ("Dostałem jedynkę ze sprawdzianu, czy mogę ją zmienić?", 4),
    ("Do której czynna jest czytelnia?", 5),
    ("Do której mogę zrezygnować z obiadu?", 6),
    ("Kiedy mogę pójść na zajęcia z programowania?", 7),
    ("Ile trwa egzamin próbny z matematyki?", 8),
    ("Skąd komórka bierze energię?", 9),
    ("Kiedy Mieszko został ochrzczony?", 10),
]

def evaluate(model_name, query_prefix="", doc_prefix=""):
    """Sprawdza, w ilu przypadkach model embeddingów wskazuje poprawny dokument jako pierwszy."""
    m = SentenceTransformer(model_name)
    d_embs = m.encode([doc_prefix + d for d in documents], normalize_embeddings=True)
    trafienia = 0
    for pytanie, poprawny in testy:
        q_emb = m.encode([query_prefix + pytanie], normalize_embeddings=True)
        wskazany = int(np.argmax(np.dot(d_embs, q_emb.T)))
        ok = wskazany == poprawny
        trafienia += ok
        print(f"  {'✅' if ok else '❌'} {pytanie}")
    print(f"→ {model_name}: {trafienia}/{len(testy)} poprawnych\n")
    return trafienia

print("Model ANGIELSKI (all-MiniLM-L6-v2):")
evaluate("sentence-transformers/all-MiniLM-L6-v2")

print("Model WIELOJĘZYCZNY (paraphrase-multilingual-MiniLM-L12-v2):")
evaluate("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Który model wygrał? Zwróć uwagę, że oba mają **podobny rozmiar** — różnica wynika z tego, **na jakich danych** je uczono.

> 🔑 **Wniosek:** RAG jest tak dobry, jak jego wyszukiwarka. Jeśli retriever poda modelowi **zły fragment**, nawet najlepszy LLM odpowie źle (albo powie „Nie wiem"), bo dostał niewłaściwą ściągę.

### 🧪 Twoja kolej – wybierz własny model embeddingów
Spróbuj kolejnych modeli i porównaj wyniki. Dobre kandydatki (mieszczą się w T4):

| Model | Uwagi |
|---|---|
| `intfloat/multilingual-e5-small` | wielojęzyczny, **wymaga prefiksów** → `query_prefix="query: "`, `doc_prefix="passage: "` |
| `intfloat/multilingual-e5-base` | j.w., większy i zwykle dokładniejszy |
| `BAAI/bge-m3` | bardzo dobry, wielojęzyczny, większy (~2 GB) |

In [ ]:
# Odkomentuj i uruchom, żeby porównać kolejny model:
# evaluate("intfloat/multilingual-e5-small", query_prefix="query: ", doc_prefix="passage: ")
# evaluate("BAAI/bge-m3")

**Dodatkowe zadanie:** dopisz **własne pytania** do listy `testy` (najlepiej takie „podchwytliwe": parafrazy, odmienione słowa, pytania bez wspólnych słów z dokumentem) i sprawdź, który model da radę. Tak właśnie inżynierowie wybierają modele do prawdziwych systemów — nie na wyczucie, tylko **testując na własnych danych**.

## 2.6 Długie dokumenty – dzielenie na fragmenty (chunking)

W prawdziwym życiu dokumenty nie są jednozdaniowe: to cała książka, regulamin, artykuł. Nie wkleisz ich w całości do promptu (model ma ograniczoną „pamięć roboczą"), a embedding całej książki byłby rozmyty.

Rozwiązanie: **tniemy tekst na krótkie fragmenty** (*chunki*) i każdy embedujemy osobno. Poniżej prosty sposób: grupy po 2 zdania.

In [ ]:
def chunk_text(text, sentences_per_chunk=2):
    """Dzieli tekst na fragmenty po kilka zdań."""
    zdania = re.split(r"(?<=[.!?])\s+", text.strip())
    return [" ".join(zdania[i:i + sentences_per_chunk]) for i in range(0, len(zdania), sentences_per_chunk)]

# 🧪 WKLEJ TU DOWOLNY DŁUŻSZY TEKST (np. fragment Wikipedii, lektury, regulaminu)
moj_tekst = """
Wisła jest najdłuższą rzeką w Polsce. Ma długość około 1047 kilometrów. Jej źródła znajdują się w Beskidzie Śląskim.
Wisła przepływa między innymi przez Kraków, Warszawę, Płock, Toruń i Gdańsk. Uchodzi do Morza Bałtyckiego.
Toruń słynie z pierników oraz z tego, że urodził się w nim Mikołaj Kopernik. Starówka Torunia znajduje się na liście UNESCO.
Pierniki toruńskie wypieka się według receptur znanych od średniowiecza. Ich ciasto dojrzewa nawet kilka miesięcy.
"""

chunki = chunk_text(moj_tekst, sentences_per_chunk=2)
chunki_emb = embed_model.encode(chunki, normalize_embeddings=True)

print(f"Tekst podzielony na {len(chunki)} fragmentów:")
for i, c in enumerate(chunki):
    print(f"  [{i}] {c[:90]}...")

pytanie = "Jak długo dojrzewa ciasto na pierniki?"
print("\n❓", pytanie)
print("💬", ask_rag(pytanie, chunki, chunki_emb, top_k=1))

### 🧪 Twoja kolej
1. Wklej własny tekst (np. fragment lektury lub artykułu) i zadaj pytania o szczegóły.
2. Zmień `sentences_per_chunk` na 1, potem na 4. Co się dzieje? (Za małe chunki → brakuje kontekstu. Za duże → rozmyty sens i dużo „szumu").
3. Zadaj pytanie, które wymaga informacji z **dwóch różnych fragmentów** — spróbuj `top_k=2`.

---

# CZĘŚĆ 3 – Podsumowanie

## Co zbudowaliśmy?

```
 PYTANIE ──► embedding pytania ──► porównanie z embeddingami dokumentów ──► najlepszy fragment
                                                                                   │
 ODPOWIEDŹ ◄── LLM ◄── prompt: „Odpowiadaj TYLKO na podstawie kontekstu" + fragment + pytanie
```

1. **Baza wiedzy** → dokumenty zamienione na wektory (embeddingi).
2. **Retriever** → znajduje fragmenty najbliższe pytaniu.
3. **Prompt** → wkleja fragmenty jako kontekst i ustawia zasady.
4. **LLM** → generuje odpowiedź oparta na dostarczonej wiedzy.

## Czego się nauczyliśmy?

* 🧠 Modele językowe **nie znają naszych danych** i potrafią zmyślać — RAG daje im „ściągę" z wiarygodnego źródła.
* 📚 **Aktualizacja wiedzy** w RAG to dopisanie dokumentu, a nie trenowanie modelu.
* 🔢 **Embedding** = tekst zamieniony na wektor; podobne znaczenie → podobne wektory.
* 🌍 **Wybór modelu embeddingów ma znaczenie** (np. język!) — i trzeba go **testować**.
* ✂️ **Chunking** — długie teksty tniemy na fragmenty.
* ⚠️ RAG nie jest magiczny: jeśli retriever poda zły fragment, odpowiedź będzie zła.

## Ograniczenia RAG – o czym warto pamiętać

* **„Śmieci na wejściu, śmieci na wyjściu"** — jeśli baza wiedzy zawiera błędy, model je powtórzy (a pewnym tonem!).
* Model **może zignorować kontekst** i odpowiedzieć z własnej pamięci, zwłaszcza gdy prompt jest słaby.
* Pytania wymagające **połączenia wielu faktów** (lub liczenia) są trudne.
* **Prywatność:** jeśli używasz modelu w chmurze, wysyłasz tam swoje dokumenty. Model uruchomiony lokalnie (jak tu, na Colabie) trzyma dane „u siebie" — dlatego RAG jest tak popularny w szpitalach, bankach i firmach.

## 💭 Pytania do dyskusji

1. Kiedy lepiej użyć RAG, a kiedy fine-tuningu? (Podpowiedź: *wiedza faktograficzna, która się zmienia* vs. *styl, format, specjalne umiejętności*.)
2. Jak sprawdzić, czy odpowiedź modelu **na pewno** pochodzi z bazy wiedzy? (Podpowiedź: wyświetlanie źródeł!)
3. Jakie **własne dane** warto by podłączyć do RAG-a: notatki z lekcji? regulamin szkoły? wiki ulubionej gry?
4. Co się stanie, gdy w bazie wiedzy ktoś umieści **fałszywą informację**? Kto powinien mieć prawo dopisywać dokumenty?

## 🚀 Zadania dodatkowe (dla chętnych)

* **Pokaż źródło:** zmodyfikuj `ask_rag`, by zwracała także numer i tekst użytego dokumentu (cytowanie źródeł).
* **Własna baza:** zbuduj RAG dla własnych notatek (np. 10–20 zdań z biologii/historii) i przetestuj na „pytaniach sprawdzianowych".
* **Test jakości:** dopisz po 10 własnych pytań do listy `testy` i wybierz najlepszy model embeddingów.
* **Zbadaj ograniczenia:** znajdź pytanie, na które RAG odpowiada **źle**, i wyjaśnij dlaczego (zły retriever? zły prompt? brak informacji?).
* **Skalowanie:** do 1000 dokumentów `np.dot` wystarczy. Co zrobić przy milionie? (Poszukaj haseł: *baza wektorowa*, *FAISS*, *ChromaDB*.)

---

**Dziękujemy za udział w warsztatach! 🎓**
Materiały możesz zmieniać i rozwijać na własną rękę — wystarczy skopiować notatnik do swojego Dysku Google (*Plik → Zapisz kopię na Dysku*).